# IDR Prediction — GAN Training
Trains an unconditional GAN on per-residue ProtBERT embeddings extracted from the DisProt training set.
Outputs `data_repository/processed/synthetic_disprot.json` for use in `train_augmented.ipynb`.

**Run all cells top to bottom.**

In [ ]:
# ── 1. Environment setup (Colab only) ────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/siavashprh/idr-prediction.git
    os.chdir('idr-prediction')
    !pip install -q -e .
    !pip install -q 'transformers==5.7.0' 'tokenizers==0.22.2' biopython omegaconf sentencepiece safetensors
    import torch
    if tuple(int(x) for x in torch.__version__.split('.')[:2]) < (2, 6):
        !pip install -q 'torch>=2.6.0'

    # Upload baseline checkpoint
    from google.colab import files
    print('Upload baseline_transformer_lstm.pth when prompted:')
    uploaded = files.upload()
    os.makedirs('data_repository/ckpt', exist_ok=True)
    for fname in uploaded:
        os.rename(fname, f'data_repository/ckpt/{fname}')

print('Working dir:', os.getcwd())

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import json, random, time
from IPython.display import clear_output
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import transformers
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

transformers.logging.set_verbosity_error()

from src.augmentation.gan import Generator, Discriminator
from src.augmentation.decoder import EmbeddingDecoder
from src.augmentation.embeddings import extract_embeddings
from src.augmentation.dataset import build_synthetic_dataset
from src.augmentation.pseudo_label import pseudo_label
from src.config import ModelConfig
from src.data.data import get_disprot_cut_data
from src.models.pre_trained_helper import get_pre_model
from src.models.sequence_models import TransformerLSTMModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Config ────────────────────────────────────────────────────────────────
from omegaconf import OmegaConf
cfg = OmegaConf.load('configs/gan.yaml')
print(OmegaConf.to_yaml(cfg))

In [ ]:
# ── 4. Load baseline model ────────────────────────────────────────────────────
# The baseline model serves two purposes:
#   (a) ProtBERT encoder for embedding extraction
#   (b) Pseudo-labeller for assigning disorder labels to synthetic sequences
print('Loading prot_bert_bfd...')
model_config = ModelConfig()
model_config.model_name = 'baseline_transformer_lstm'

baseline = TransformerLSTMModel(
    pre_model_name='Rostlab/prot_bert_bfd',
    device=device,
    input_dim=1024, linear_hidden_dim=64,
    num_heads=4, num_blocks=2, dropout=0.6,
    model_config=model_config,
    with_lstm=True, lstm_n_layers=2,
)
baseline.load()
baseline.model.eval()
print('Baseline model loaded.')

In [ ]:
# ── 5. Extract ProtBERT embeddings ───────────────────────────────────────────
emb_path = cfg.data.embeddings_path

if Path(emb_path).exists():
    print(f'Loading cached embeddings from {emb_path}')
    embeddings = torch.load(emb_path, map_location='cpu')
else:
    print('Extracting embeddings (runs once, ~10 min on T4)...')
    pre_model = baseline.model.pre_model
    pre_tokenizer = baseline.model.pre_tokenizer
    embeddings = extract_embeddings(pre_model, pre_tokenizer, device, emb_path)

# Flatten to individual residue vectors for GAN training
all_embs = torch.cat(embeddings, dim=0).float()  # (N_total_residues, 1024)
print(f'Total residue embeddings: {all_embs.shape[0]:,}  dim={all_embs.shape[1]}')

In [ ]:
# ── 6. Train GAN ─────────────────────────────────────────────────────────────
G = Generator(latent_dim=cfg.gan.latent_dim).to(device)
D = Discriminator().to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=cfg.gan.lr, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=cfg.gan.lr, betas=(0.5, 0.999))
bce = nn.BCELoss()

batch_size = cfg.gan.batch_size
n_epoch    = cfg.gan.n_epoch
n          = len(all_embs)

g_losses, d_losses = [], []
t_start = time.time()

for epoch in range(n_epoch):
    G.train(); D.train()
    perm = torch.randperm(n)
    epoch_g, epoch_d = [], []

    for start in range(0, n - batch_size, batch_size):
        real = all_embs[perm[start : start + batch_size]].to(device)
        bs   = real.size(0)

        # ── Discriminator step ───────────────────────────────────────────────
        z      = torch.randn(bs, cfg.gan.latent_dim, device=device)
        fake   = G(z).detach()
        r_lab  = torch.full((bs, 1), cfg.gan.real_label_smooth, device=device)
        f_lab  = torch.zeros(bs, 1, device=device)

        d_loss = bce(D(real), r_lab) + bce(D(fake), f_lab)
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

        # ── Generator step ───────────────────────────────────────────────────
        z      = torch.randn(bs, cfg.gan.latent_dim, device=device)
        g_loss = bce(D(G(z)), torch.ones(bs, 1, device=device))
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        epoch_g.append(g_loss.item())
        epoch_d.append(d_loss.item())

    g_losses.append(np.mean(epoch_g))
    d_losses.append(np.mean(epoch_d))
    elapsed = (time.time() - t_start) / 60

    clear_output(wait=True)
    print(f'Epoch {epoch+1}/{n_epoch}  G={g_losses[-1]:.4f}  D={d_losses[-1]:.4f}  ({elapsed:.1f} min)')

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(g_losses, label='G loss', color='steelblue')
    ax.plot(d_losses, label='D loss', color='crimson')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.set_title(f'GAN training — epoch {epoch+1}/{n_epoch}')
    ax.legend(); plt.tight_layout(); plt.show(); plt.close()

print(f'\nGAN training complete in {(time.time()-t_start)/60:.1f} min')

# Save checkpoints
os.makedirs('data_repository/ckpt', exist_ok=True)
torch.save(G.state_dict(), 'data_repository/ckpt/gan_generator.pth')
torch.save(D.state_dict(), 'data_repository/ckpt/gan_discriminator.pth')
print('Checkpoints saved.')

In [ ]:
# ── 7. GAN Evaluation ────────────────────────────────────────────────────────
G.eval()

# (a) Loss curves (final)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(g_losses, label='G loss', color='steelblue')
ax.plot(d_losses, label='D loss', color='crimson')
ax.set_xlabel('epoch'); ax.set_ylabel('loss')
ax.set_title('GAN training loss (final)')
ax.legend(); plt.tight_layout(); plt.show(); plt.close()

# (b) t-SNE: real vs generated embeddings
n_vis = 2000
real_sample = all_embs[torch.randperm(len(all_embs))[:n_vis]].numpy()
with torch.no_grad():
    z_vis = torch.randn(n_vis, cfg.gan.latent_dim, device=device)
    fake_sample = G(z_vis).cpu().numpy()

combined = np.vstack([real_sample, fake_sample])
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
proj = tsne.fit_transform(combined)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(proj[:n_vis, 0], proj[:n_vis, 1], s=3, alpha=0.4, label='real', color='steelblue')
ax.scatter(proj[n_vis:, 0], proj[n_vis:, 1], s=3, alpha=0.4, label='generated', color='crimson')
ax.set_title('t-SNE: real vs generated embeddings')
ax.legend(); plt.tight_layout(); plt.show(); plt.close()

# (c) Pseudo-label disorder rate
print('\nEstimating pseudo-label disorder rate on 100 generated sequences...')
decoder = EmbeddingDecoder()
sample_seqs = []
with torch.no_grad():
    for _ in range(100):
        z = torch.randn(50, cfg.gan.latent_dim, device=device)
        embs = G(z)
        sample_seqs.append(decoder.decode_sequence(embs.cpu()))

sample_labels = pseudo_label(sample_seqs, baseline)
total_res = sum(len(l) for l in sample_labels)
disordered = sum(sum(l) for l in sample_labels)
print(f'Generated disorder rate : {100*disordered/total_res:.1f}%  (real DisProt: ~16%)')
print('A rate << 16% confirms the unconditional GAN is order-biased (expected).')

In [ ]:
# ── 8. Generate synthetic dataset ────────────────────────────────────────────
print(f'Generating {cfg.data.n_synthetic_chunks} synthetic sequences of length {cfg.data.chunk_len}...')

data = build_synthetic_dataset(
    n_chunks  = cfg.data.n_synthetic_chunks,
    chunk_len = cfg.data.chunk_len,
    generator = G,
    decoder   = EmbeddingDecoder(),
    model     = baseline,
    device    = device,
    save_path = cfg.data.synthetic_path,
    latent_dim= cfg.gan.latent_dim,
)

print(f'Synthetic sequences : {len(data["sequences"])}')
disord = sum(sum(l) for l in data["disorder region"])
total  = sum(len(l) for l in data["disorder region"])
print(f'Disorder rate       : {100*disord/total:.1f}%')
print(f'Saved to            : {cfg.data.synthetic_path}')

if IN_COLAB:
    from google.colab import files
    files.download(cfg.data.synthetic_path)
    print('Browser download triggered.')